# BarcodeQuest - 유화풍 유럽 배경 에셋 생성기

**사용법:**
1. 런타임 > 런타임 유형 변경 > **T4 GPU** 선택
2. 셀을 위에서 아래로 순서대로 실행
3. 마지막 셀에서 zip 파일 다운로드

---

## 1. 설치 및 환경 설정

In [ ]:
!pip install -q diffusers transformers accelerate safetensors
!pip install -q invisible-watermark>=0.2.0
# torchvision을 현재 torch 버전에 맞게 재설치
!pip install -q --upgrade torchvision torchaudio
!pip install -q xformers

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1024**3:.1f} GB")

## 2. 모델 로드 (SDXL)

In [ ]:
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
import torch

model_id = "stabilityai/stable-diffusion-xl-base-1.0"

pipe = StableDiffusionXLPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_xformers_memory_efficient_attention()
pipe = pipe.to("cuda")

print("Model loaded!")

## 3. 프롬프트 정의

모든 배경 에셋의 프롬프트가 여기에 정의되어 있습니다.  
특정 이미지만 생성하려면 `PROMPTS` 딕셔너리에서 원하는 항목만 남기세요.

In [ ]:
# --- 공통 스타일 ---
STYLE = (
    "classical oil painting, thick impasto brushstrokes, "
    "rich vibrant oil colors on canvas, visible paint texture, "
    "reminiscent of Caspar David Friedrich and William Turner, "
    "dramatic natural lighting, breathtaking vista, "
    "masterpiece, museum quality, best quality"
)

NEGATIVE = (
    "text, watermark, signature, logo, ui elements, blurry, "
    "low quality, worst quality, jpeg artifacts, deformed, "
    "ugly, duplicate, photorealistic, 3d render, cartoon, anime, "
    "buildings, city, village, people, characters, fantasy"
)

def prompt(scene_desc):
    return f"{scene_desc}, {STYLE}"

# --- 전체 프롬프트: 압도적 자연 절경 시리즈 ---
PROMPTS = {

    # ============================================
    #  바다 / 해안 절경
    # ============================================
    "bg_sea_golden_cliffs": prompt(
        "Majestic golden hour coastal cliffs overlooking endless ocean, "
        "towering sea stacks rising from turquoise waves, "
        "warm amber sunlight painting the cliff faces, "
        "white foam crashing against ancient rocks, "
        "vast open sky with streaked golden clouds"
    ),
    "bg_sea_sunrise_horizon": prompt(
        "Breathtaking ocean sunrise with crimson and gold sky, "
        "perfectly still mirror-like water reflecting the blazing horizon, "
        "silhouetted distant islands, delicate pink clouds, "
        "the first light of dawn spreading across infinite sea"
    ),
    "bg_sea_stormy_coast": prompt(
        "Dramatic stormy seascape with massive waves crashing against rocks, "
        "dark brooding clouds with rays of light breaking through, "
        "wild untamed ocean power, spray and mist in the air, "
        "Turner-esque maritime drama, deep greens and greys"
    ),
    "bg_sea_moonlit_bay": prompt(
        "Serene moonlit bay at midnight, full moon reflected as "
        "a shimmering silver path across calm dark waters, "
        "gentle waves lapping against sandy shore, "
        "starry sky fading into deep indigo, "
        "peaceful solitude, luminous nocturnal beauty"
    ),

    # ============================================
    #  설산 / 알프스 절경
    # ============================================
    "bg_mountain_sunrise_peak": prompt(
        "Awe-inspiring snow-capped mountain peak at sunrise, "
        "alpenglow painting the summit in rose gold and pink, "
        "sea of clouds filling the valleys below, "
        "pristine white snow contrasting against deep blue sky, "
        "overwhelming scale and majesty of nature"
    ),
    "bg_mountain_glacial_lake": prompt(
        "Crystal clear glacial lake reflecting snow-covered mountains, "
        "perfect mirror reflection in still turquoise water, "
        "surrounded by rugged alpine terrain, "
        "bright midday sun illuminating every detail, "
        "pristine untouched wilderness"
    ),
    "bg_mountain_golden_valley": prompt(
        "Vast alpine valley in golden autumn light, "
        "snow-dusted peaks towering above golden larch forests, "
        "a winding river of pure blue cutting through the valley, "
        "warm afternoon light creating long dramatic shadows, "
        "grand sweeping panoramic landscape"
    ),
    "bg_mountain_starry_night": prompt(
        "Snow-covered mountain range under the Milky Way, "
        "millions of stars filling the crystal clear night sky, "
        "faint aurora borealis dancing in green and purple, "
        "moonlit snow glowing with soft blue light, "
        "cosmic vastness above silent frozen peaks"
    ),

    # ============================================
    #  초원 / 대자연
    # ============================================
    "bg_meadow_endless_green": prompt(
        "Vast endless rolling green meadows stretching to the horizon, "
        "dramatic cumulus clouds casting moving shadows on the land, "
        "wildflowers dotting the lush grass in yellow and purple, "
        "a single ancient tree standing majestically, "
        "overwhelming sense of freedom and open space"
    ),
    "bg_meadow_lavender_sunset": prompt(
        "Infinite lavender fields at golden sunset, "
        "rows of deep purple flowers stretching to distant mountains, "
        "warm amber sky melting into soft violet, "
        "gentle warm breeze visible in the swaying flowers, "
        "Provencal countryside in peak summer bloom"
    ),
    "bg_meadow_misty_morning": prompt(
        "Ethereal misty morning over a vast green valley, "
        "layers of soft white fog nestled between rolling hills, "
        "golden sunrise breaking through the mist, "
        "dew-covered grass sparkling like diamonds, "
        "dreamlike tranquil atmosphere, depth and mystery"
    ),

    # ============================================
    #  야경 / 밤하늘
    # ============================================
    "bg_night_aurora_lake": prompt(
        "Spectacular aurora borealis over a still mountain lake, "
        "vivid green and purple northern lights dancing across the sky, "
        "perfect reflection in the mirror-like dark water, "
        "snow-covered shores, scattered bright stars, "
        "overwhelming celestial beauty"
    ),
    "bg_night_starry_canyon": prompt(
        "Deep canyon under a star-filled sky, "
        "the Milky Way arching brilliantly across the heavens, "
        "warm sandstone canyon walls glowing faintly, "
        "infinite depth of stars and nebulae visible, "
        "profound silence and cosmic grandeur"
    ),
    "bg_night_moonrise_ocean": prompt(
        "Giant golden moon rising over dark ocean horizon, "
        "moonlight creating a blazing golden path on black water, "
        "scattered clouds catching warm lunar glow, "
        "vast dark sky transitioning from warm gold to deep blue, "
        "romantic and majestic nocturnal seascape"
    ),

    # ============================================
    #  숲 / 계곡
    # ============================================
    "bg_forest_sunbeam_ancient": prompt(
        "Ancient primeval forest with massive towering trees, "
        "dramatic sunbeams piercing through the dense canopy, "
        "shafts of golden light illuminating mossy forest floor, "
        "ferns and ancient roots creating depth and texture, "
        "cathedral-like natural grandeur"
    ),
    "bg_forest_autumn_river": prompt(
        "Stunning autumn river valley with fiery red and gold foliage, "
        "crystal clear river winding through colorful forest, "
        "warm afternoon light making leaves glow like flames, "
        "moss-covered boulders in the stream, "
        "peak autumn color in full glory"
    ),
    "bg_forest_waterfall_paradise": prompt(
        "Majestic multi-tiered waterfall in lush green ravine, "
        "cascading white water tumbling over mossy rocks, "
        "rainbow mist catching sunlight at the base, "
        "surrounded by verdant tropical-like vegetation, "
        "paradise-like natural wonder, raw power and beauty"
    ),

    # ============================================
    #  특별 장면
    # ============================================
    "bg_special_volcanic_dawn": prompt(
        "Dramatic volcanic landscape at dawn, "
        "smoldering peak with wisps of steam against orange sky, "
        "dark lava fields contrasting with golden sunrise, "
        "raw primal power of the earth, "
        "sublime and terrifying natural beauty"
    ),
}

print(f"Total prompts: {len(PROMPTS)}")
for name in PROMPTS:
    print(f"  - {name}")

## 4. 테스트 생성 (메인 로비 1장)

먼저 1장만 생성해서 스타일을 확인합니다.  
마음에 들면 다음 셀에서 전체 배치를 실행하세요.

In [ ]:
import os
from IPython.display import display

os.makedirs("/content/barcode_backgrounds", exist_ok=True)

# --- 테스트: 황금빛 해안 절벽 1장 ---
test_name = "bg_sea_golden_cliffs"
test_prompt = PROMPTS[test_name]

print(f"Generating: {test_name}")
print(f"Prompt: {test_prompt[:120]}...")

image = pipe(
    prompt=test_prompt,
    negative_prompt=NEGATIVE,
    width=1344,   # 16:9 ratio for SDXL
    height=768,
    num_inference_steps=35,
    guidance_scale=7.5,
    generator=torch.Generator("cuda").manual_seed(42),
).images[0]

path = f"/content/barcode_backgrounds/{test_name}.png"
image.save(path)
print(f"Saved: {path}")
display(image)

## 5. 시드 조정 (선택사항)

테스트 이미지가 마음에 안 들면 시드를 바꿔서 다시 생성해보세요.

In [ ]:
#@title 시드 변경 후 재생성 { run: "auto" }
SEED = 123  #@param {type:"integer"}

image = pipe(
    prompt=PROMPTS["bg_lobby_main"],
    negative_prompt=NEGATIVE,
    width=1344,
    height=768,
    num_inference_steps=35,
    guidance_scale=7.5,
    generator=torch.Generator("cuda").manual_seed(SEED),
).images[0]

image.save(f"/content/barcode_backgrounds/bg_lobby_main_seed{SEED}.png")
display(image)

## 6. 전체 배치 생성

모든 배경을 한 번에 생성합니다. (약 20~30분 소요)

In [ ]:
import time

BATCH_SEED = 42  # 전체 배치에 사용할 기본 시드
OUTPUT_DIR = "/content/barcode_backgrounds"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 이미 생성된 파일은 건너뛰기
existing = set(os.listdir(OUTPUT_DIR))

total = len(PROMPTS)
start_time = time.time()

for i, (name, p) in enumerate(PROMPTS.items(), 1):
    filename = f"{name}.png"
    if filename in existing:
        print(f"[{i}/{total}] SKIP (exists): {name}")
        continue

    print(f"[{i}/{total}] Generating: {name}...")
    t0 = time.time()

    # UI 배경은 4:3, 나머지는 16:9
    if "_ui_" in name:
        w, h = 1152, 896
    else:
        w, h = 1344, 768

    image = pipe(
        prompt=p,
        negative_prompt=NEGATIVE,
        width=w,
        height=h,
        num_inference_steps=35,
        guidance_scale=7.5,
        generator=torch.Generator("cuda").manual_seed(BATCH_SEED + i),
    ).images[0]

    image.save(os.path.join(OUTPUT_DIR, filename))
    elapsed = time.time() - t0
    total_elapsed = time.time() - start_time
    print(f"  Done in {elapsed:.1f}s | Total: {total_elapsed:.0f}s")

print(f"\nAll done! {total} images generated in {time.time()-start_time:.0f}s")
print(f"Output: {OUTPUT_DIR}")

## 7. 결과 미리보기

In [ ]:
from PIL import Image
from IPython.display import display, HTML
import glob

files = sorted(glob.glob(f"{OUTPUT_DIR}/bg_*.png"))
print(f"Generated {len(files)} images:\n")

for f in files:
    name = os.path.basename(f).replace(".png", "")
    img = Image.open(f)
    # 미리보기용 리사이즈
    preview = img.copy()
    preview.thumbnail((672, 384))
    print(f"--- {name} ({img.width}x{img.height}) ---")
    display(preview)

## 8. ZIP 다운로드

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/barcode_backgrounds"
shutil.make_archive(zip_path, 'zip', zip_path)

print(f"Downloading: barcode_backgrounds.zip")
files.download(f"{zip_path}.zip")

## 9. 개별 재생성 (선택사항)

특정 이미지만 다시 생성하고 싶을 때 사용하세요.

In [ ]:
#@title 개별 재생성 { run: "auto" }
TARGET = "bg_battle_forest_morning"  #@param {type:"string"}
SEED = 42  #@param {type:"integer"}
STEPS = 35  #@param {type:"slider", min:20, max:50, step:5}
GUIDANCE = 7.5  #@param {type:"slider", min:5, max:12, step:0.5}

if TARGET not in PROMPTS:
    print(f"Invalid target. Choose from: {list(PROMPTS.keys())}")
else:
    if "_ui_" in TARGET:
        w, h = 1152, 896
    else:
        w, h = 1344, 768

    image = pipe(
        prompt=PROMPTS[TARGET],
        negative_prompt=NEGATIVE,
        width=w,
        height=h,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        generator=torch.Generator("cuda").manual_seed(SEED),
    ).images[0]

    image.save(f"{OUTPUT_DIR}/{TARGET}.png")
    print(f"Regenerated: {TARGET} (seed={SEED})")
    display(image)